In [13]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score

pd.set_option('display.max_columns', None)

In [14]:
df_clean = pd.read_pickle('data/clean_transactions.pkl')
print(df_clean.shape)

# The last date in the data. Everything is measured relative to it.
max_date = df_clean['InvoiceDate'].max()
print(max_date)                        

HORIZON = 90        # "will the customer buy again within 90 days?"
QUIET_DAYS = 60     # a customer is "quiet" after 60 days without a purchase (see Part C)

(805549, 9)
2011-12-09 12:50:00


**Helper Function**

In [15]:
def build_dataset(df, cutoff, horizon_days):
    past = df[df['InvoiceDate'] <= cutoff]
    future = df[(df['InvoiceDate'] > cutoff) &
                (df['InvoiceDate'] <= cutoff + pd.Timedelta(days=horizon_days))]
    feat = past.groupby('Customer ID').agg(
        Recency=('InvoiceDate', lambda x: (cutoff - x.max()).days),
        Frequency=('Invoice', 'nunique'),
        Monetary=('TotalPrice', 'sum'),
        FirstPurchase=('InvoiceDate', 'min')).reset_index()
    feat['label'] = feat['Customer ID'].isin(set(future['Customer ID'])).astype(int)
    return feat

def prep(feat, cap=None):
    spend = feat['Monetary'].clip(upper=cap) if cap else feat['Monetary']
    return pd.DataFrame({'Recency': feat['Recency'],
                         'log_Frequency': np.log1p(feat['Frequency']),
                         'log_Monetary': np.log1p(spend)})

**Out-of-time test**

In [16]:
def run(horizon=90, winsorize=False):
    test_cutoff = max_date - pd.Timedelta(days=horizon)
    train_cutoff = test_cutoff - pd.Timedelta(days=horizon)
    train = build_dataset(df_clean, train_cutoff, horizon)
    test = build_dataset(df_clean, test_cutoff, horizon)
    cap = train['Monetary'].quantile(0.99) if winsorize else None
    X_train, X_test = prep(train, cap), prep(test, cap)
    scaler = StandardScaler().fit(X_train)
    model = LogisticRegression(random_state=42).fit(scaler.transform(X_train), train['label'])
    test['p'] = model.predict_proba(scaler.transform(X_test))[:, 1]
    pred = (test['p'] >= 0.5).astype(int)
    naive = (test['Recency'] <= train['Recency'].median()).astype(int)   # "recent buyers will return"
    y = test['label']
    res = {'window': horizon,
           'train_rate': round(train['label'].mean(), 3), 'test_rate': round(y.mean(), 3),
           'model_acc': round(accuracy_score(y, pred), 3),
           'naive_acc': round(accuracy_score(y, naive), 3),
           'majority_acc': round(1 - y.mean(), 3),
           'model_auc': round(roc_auc_score(y, test['p']), 3),
           'naive_auc': round(roc_auc_score(y, -test['Recency']), 3),
           'precision': round(precision_score(y, pred), 3),
           'recall': round(recall_score(y, pred), 3)}
    return res, test

In [17]:
# B1. Main result (90-day window)
res90, test90 = run(90)
res90

{'window': 90,
 'train_rate': np.float64(0.321),
 'test_rate': np.float64(0.434),
 'model_acc': 0.709,
 'naive_acc': 0.704,
 'majority_acc': np.float64(0.566),
 'model_auc': 0.793,
 'naive_auc': 0.763,
 'precision': 0.786,
 'recall': 0.454}

In [18]:
# B2. Do the conclusions change with a 60-, 90-, or 120-day window?
pd.DataFrame([run(h)[0] for h in (60, 90, 120)])

,window,train_rate,test_rate,model_acc,naive_acc,majority_acc,model_auc,naive_auc,precision,recall
0,60,0.293,0.369,0.732,0.684,0.631,0.784,0.750,0.719,0.449
1,90,0.321,0.434,0.709,0.704,0.566,0.793,0.763,0.786,0.454
2,120,0.393,0.468,0.724,0.715,0.532,0.807,0.777,0.784,0.566


In [19]:
# B3. Outlier check: cap spend at the training set's 99th percentile. Do results move?
run(90, winsorize=True)[0]

{'window': 90,
 'train_rate': np.float64(0.321),
 'test_rate': np.float64(0.434),
 'model_acc': 0.709,
 'naive_acc': 0.704,
 'majority_acc': np.float64(0.566),
 'model_auc': 0.793,
 'naive_auc': 0.763,
 'precision': 0.785,
 'recall': 0.455}

In [20]:
# B4. Does a higher score mean a higher repeat rate? (decile 10 = highest scores)
test90['decile'] = pd.qcut(test90['p'].rank(method='first'), 10, labels=range(1, 11))
print(test90.groupby('decile', observed=True)['label'].agg(['count', 'mean']).round(3))

k = int(len(test90) * 0.10)
print('Repeat rate, top 10% by model score:      ', round(test90.nlargest(k, 'p')['label'].mean(), 3))
print('Repeat rate, 10% most recent buyers only: ', round(test90.nsmallest(k, 'Recency')['label'].mean(), 3))

        count   mean
decile              
1         529  0.079
2         528  0.169
3         528  0.220
4         528  0.292
5         528  0.316
6         528  0.483
7         527  0.514
8         529  0.609
9         528  0.748
10        528  0.911
Repeat rate, top 10% by model score:       0.911
Repeat rate, 10% most recent buyers only:  0.777


## What counts as "gone quiet"?

In [21]:
inv = (df_clean.groupby(['Customer ID', 'Invoice'])['InvoiceDate'].min()
       .reset_index().sort_values(['Customer ID', 'InvoiceDate']))
gaps = inv.groupby('Customer ID')['InvoiceDate'].diff().dt.days.dropna()
print('median gap:', gaps.median(), '| 75th percentile:', gaps.quantile(0.75)) 

median gap: 24.0 | 75th percentile: 61.0


## Train the final model and score every customer

In [12]:
train_final = build_dataset(df_clean, max_date - pd.Timedelta(days=HORIZON), HORIZON)
X_f = prep(train_final)
scaler_f = StandardScaler().fit(X_f)
model_f = LogisticRegression(random_state=42).fit(scaler_f.transform(X_f), train_final['label'])
print(dict(zip(X_f.columns, model_f.coef_[0].round(3))))   # Recency negative; Frequency, Monetary positive

scored = build_dataset(df_clean, max_date, HORIZON).drop(columns='label')
scored['p_return'] = model_f.predict_proba(scaler_f.transform(prep(scored)))[:, 1]
print(len(scored), round(scored['p_return'].mean(), 3)) 

{'Recency': np.float64(-0.748), 'log_Frequency': np.float64(0.468), 'log_Monetary': np.float64(0.304)}
5878 0.459
